Importing all needed libraries for the notebook

In [15]:
import os
import findspark
from pyspark.sql import SparkSession


In [2]:
findspark.init()

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Local Laptop") \
    .master("local[*]") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

spark

In [4]:
DEC_file_path = "Data/US_Accidents_Dec21.csv"
MAR_file_path = "Data/US_Accidents_Mar23.csv"

In [5]:
Dec_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("mode", "DROPMALFORMED") \
    .option("multiLine", True) \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("ignoreLeadingWhiteSpace", True) \
    .option("ignoreTrailingWhiteSpace", True) \
    .load(DEC_file_path)

In [6]:
MAR_df = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("mode", "DROPMALFORMED") \
    .option("multiLine", True) \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("ignoreLeadingWhiteSpace", True) \
    .option("ignoreTrailingWhiteSpace", True) \
    .load(MAR_file_path)

In [7]:
Dec_df = Dec_df.repartition(16)
MAR_df = MAR_df.repartition(16)

Drop unwanted columns

In [8]:
df_dec = Dec_df.drop("Number", "Side")
df_mar = MAR_df.drop("Source")

Align schemas automatically

In [9]:
df_merged = df_dec.unionByName(df_mar, allowMissingColumns=True)

Remove duplicates

In [10]:
df_merged = df_merged.dropDuplicates()

Optimize partitions

In [11]:
df_merged = df_merged.repartition(16)

Write as Parquet

In [12]:
output_path = "Data/Full_Data"

In [13]:
df_merged.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(output_path)

Showing all files in the parquet dire

In [16]:
for file in os.listdir(output_path):
    if file.endswith(".parquet"):
        print(file)

part-00000-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00001-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00002-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00003-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00004-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00005-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00006-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00007-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00008-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00009-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00010-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00011-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00012-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00013-1daaa805-6bbd-4513-aacc-f06f0bf717b2-c000.snappy.parquet
part-00014-1daaa805-6bbd-4513-aacc-f06f0bf717b2-